# Laboratorio 4 — Regresión
## Analítica de Datos · Universidad de Antioquia · Instituto de Matemáticas
**Profesor:** Duván Cataño

---

**Objetivo:** Desarrollar un pipeline completo de modelado para un problema de regresión,
aplicando múltiples algoritmos de Analítica de Datos y comparando su desempeño mediante
métricas apropiadas.

**Correcciones aplicadas respecto al script original** (`reporte_errores.md`):
| ID | Severidad | Descripción |
|---|---|---|
| L4-1 | 🔴 Crítico | Rutas relativas inconsistentes → rutas absolutas con `pathlib` |
| L4-2 | 🔴 Crítico | LightGBM etiquetado como "Random Forest" en 3 lugares |
| L4-3 | 🟡 Moderado | Feature importances tomadas del preprocesador de Ridge → del de LightGBM |
| L4-4 | 🟡 Moderado | `cross_val_score` redundante → R² extraído de `cv_results_` |
| L4-5 | 🟡 Observación | Variables predictoras adicionales incluidas si existen en el dataset |

## 1. Comprensión del Problema

- **Variable objetivo:** `price` — precio por noche en USD de un alojamiento en Airbnb.
- **Tipo de regresión:** Regresión continua multivariada. La variable objetivo es un número
  real positivo (precio en dólares), no una categoría.
- **Contexto:** El dataset contiene listados de Airbnb con variables geográficas (latitud,
  longitud), de actividad (reseñas, disponibilidad) y de categoría (tipo de habitación).
  El modelo debe aprender a estimar el precio a partir de estas características.

> **Nota metodológica:** `IsotonicRegression` exige entrada unidimensional y no es compatible
> con un pipeline multivariado estándar. Se documenta su exclusión como justificación explícita
> del ítem 3d del enunciado.

## Importaciones y Configuración de Rutas

**Corrección L4-1:** El script original mezclaba dos convenciones de ruta incompatibles.
Aquí usamos `pathlib.Path` con rutas absolutas derivadas de la ubicación real del notebook,
eliminando toda dependencia del directorio de trabajo (CWD) desde el que se lance Jupyter.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from pathlib import Path

from sklearn.model_selection import (
    train_test_split, cross_validate, KFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ── FIX L4-1: Rutas absolutas derivadas de la ubicación del notebook ──────
# El notebook reside en: .../ml-proyecto_analitica_datos/notebooks/
# CWD en VS Code / Jupyter = directorio del notebook → .parent = PROJECT_DIR
_notebook_dir = Path(os.path.abspath(''))
PROJECT_DIR = _notebook_dir.parent if _notebook_dir.name == 'notebooks' else _notebook_dir

DATA_PATH   = PROJECT_DIR / 'data' / 'raw' / 'dataset_regresion_listings.csv'
REPORTS_DIR = PROJECT_DIR / 'reports' / 'lab4'
MODELS_DIR  = PROJECT_DIR / 'models'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = '#2563EB'

print('=' * 60)
print('  LAB 4 — REGRESIÓN — ANALÍTICA DE DATOS — UdeA')
print('=' * 60)
print(f'  PROJECT_DIR : {PROJECT_DIR}')
print(f'  DATA_PATH   : {DATA_PATH}')
print(f'  REPORTS_DIR : {REPORTS_DIR}')
print(f'  MODELS_DIR  : {MODELS_DIR}')

## 2. Carga y Limpieza de Datos

El dataset de Airbnb contiene el precio como texto con símbolo `$` y comas (`$1,200.00`).
Se aplica limpieza, se eliminan precios nulos/negativos y se recortan outliers extremos
usando el percentil 99 (solo el 1% más caro es descartado).

In [ ]:
print(f'[1/8] Cargando dataset desde: {DATA_PATH}')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'      Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'\nColumnas disponibles ({len(df.columns)}):')
print(list(df.columns))

print('\n--- Primeras 3 filas ---')
display(df.head(3))

# Tipos de datos y valores faltantes
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'nulos': df.isnull().sum(),
    '% nulos': (df.isnull().sum() / len(df) * 100).round(2)
})
print('\n--- Valores faltantes ---')
display(info_df[info_df['nulos'] > 0])

In [ ]:
print('[2/8] Limpiando variable objetivo price...')

if df['price'].dtype == object:
    df['price'] = (
        df['price']
        .astype(str)
        .str.replace(r'[\$,\s]', '', regex=True)
        .replace('', np.nan)
        .astype(float)
    )

n_antes = len(df)
df = df[df['price'] > 0].dropna(subset=['price'])
p99 = df['price'].quantile(0.99)
df = df[df['price'] <= p99]

print(f'  Filas eliminadas (precio inválido + outliers p99): {n_antes - len(df):,}')
print(f'  Precio máximo tras filtro p99: ${p99:.2f}')
print(f'  Filas restantes: {len(df):,}')

print(f'\n  Estadísticas de price:')
display(df['price'].describe().rename('price').to_frame().T.round(2))

## 2b. Selección de Variables Predictoras

**Corrección L4-5 (observación):** El script original omitía variables de alto impacto
en el precio de Airbnb como `neighbourhood_group`, `neighbourhood` y `minimum_nights`.
Se añade detección dinámica: si existen en el dataset se incluyen automáticamente.

| Variable | Tipo | Justificación |
|---|---|---|
| `latitude`, `longitude` | Numérica | Posición geográfica exacta |
| `number_of_reviews` | Numérica | Indicador de popularidad |
| `reviews_per_month` | Numérica | Velocidad de reseñas (demanda) |
| `availability_365` | Numérica | Disponibilidad anual |
| `minimum_nights` | Numérica | Restricción mínima (si existe) |
| `calculated_host_listings_count` | Numérica | Profesionalidad del host (si existe) |
| `room_type` | Categórica | Tipo de habitación (gran impacto en precio) |
| `neighbourhood_group` | Categórica | Zona geográfica amplia (si existe) |
| `neighbourhood` | Categórica | Barrio específico (si existe) |

In [ ]:
# Variables base (siempre presentes en el dataset de Airbnb NYC)
VARIABLES_NUMERICAS_BASE = [
    'latitude', 'longitude',
    'number_of_reviews', 'reviews_per_month', 'availability_365'
]
VARIABLES_CATEGORICAS_BASE = ['room_type']

# FIX L4-5: Columnas adicionales de alto impacto — incluir si existen
COLS_NUM_EXTRA = ['minimum_nights', 'calculated_host_listings_count']
COLS_CAT_EXTRA = ['neighbourhood_group', 'neighbourhood']

VARIABLES_NUMERICAS   = VARIABLES_NUMERICAS_BASE + [c for c in COLS_NUM_EXTRA if c in df.columns]
VARIABLES_CATEGORICAS = VARIABLES_CATEGORICAS_BASE + [c for c in COLS_CAT_EXTRA if c in df.columns]
VARIABLE_OBJETIVO     = 'price'

print(f'Variables numéricas   ({len(VARIABLES_NUMERICAS)}): {VARIABLES_NUMERICAS}')
print(f'Variables categóricas ({len(VARIABLES_CATEGORICAS)}): {VARIABLES_CATEGORICAS}')

cols_necesarias = VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS + [VARIABLE_OBJETIVO]
cols_faltantes  = [c for c in cols_necesarias if c not in df.columns]
if cols_faltantes:
    raise ValueError(f'Columnas no encontradas en el dataset: {cols_faltantes}\n'
                     f'Columnas disponibles: {list(df.columns)}')

df = df[cols_necesarias].copy()
print(f'\nDataset final: {df.shape[0]:,} filas × {df.shape[1]} columnas')

## 3. Preparación de los Datos

Se construye un `Pipeline` de scikit-learn para encapsular todo el preprocesamiento.
Esto garantiza que las transformaciones (imputación, escalamiento, codificación)
se apliquen **solo sobre los datos de entrenamiento** en cada fold de la validación
cruzada, evitando *data leakage*.

| Paso | Variables | Transformación |
|---|---|---|
| `SimpleImputer(median)` | Numéricas | Rellena NaN con la mediana |
| `StandardScaler` | Numéricas | Media=0, Std=1 (necesario para Ridge/LASSO) |
| `SimpleImputer(most_frequent)` | Categóricas | Rellena NaN con la moda |
| `OneHotEncoder` | Categóricas | Codificación 0/1, ignora categorías nuevas |

In [ ]:
X = df[VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS]
y = df[VARIABLE_OBJETIVO]

# División 70 % train / 30 % test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print(f'Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas')

# Preprocesador numérico
preprocesador_numerico = Pipeline(steps=[
    ('imputar',  SimpleImputer(strategy='median')),
    ('escalar',  StandardScaler())
])

# Preprocesador categórico
preprocesador_categorico = Pipeline(steps=[
    ('imputar',   SimpleImputer(strategy='most_frequent')),
    ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer: aplica cada preprocesador a sus columnas correspondientes
preprocesador = ColumnTransformer(transformers=[
    ('num', preprocesador_numerico,   VARIABLES_NUMERICAS),
    ('cat', preprocesador_categorico, VARIABLES_CATEGORICAS)
])

print('Preprocesador construido correctamente.')

## 4. Definición de Modelos de Regresión

Se entrenan 7 modelos (IsotonicRegression excluida — requiere entrada 1D univariada,
incompatible con pipeline multivariado; ítem 3d del enunciado).

| Modelo | Tipo | Característica principal |
|---|---|---|
| Regresión Lineal | Lineal | Base de referencia, interpretable |
| Ridge | Lineal | Penalización L2, reduce sobreajuste |
| LASSO | Lineal | Penalización L1, selección automática de variables |
| Árbol de Decisión | No lineal | Reglas binarias, interpretable |
| Random Forest | No lineal | Ensamble de árboles, robusto |
| XGBoost | No lineal | Boosting secuencial, alta precisión |
| LightGBM | No lineal | Boosting eficiente en memoria y velocidad |

In [ ]:
print('[4/8] Definiendo modelos de regresión...')

MODELOS = {
    'Regresión Lineal': LinearRegression(),
    'Ridge':  Ridge(alpha=1.0),
    'LASSO':  Lasso(alpha=1.0, max_iter=5000),
    'Árbol de Decisión': DecisionTreeRegressor(random_state=42, max_depth=8),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=6,
        random_state=42, n_jobs=-1, verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=100, learning_rate=0.1,
        random_state=42, n_jobs=-1, verbose=-1
    ),
}

# Envolver cada modelo en pipeline completo: preprocesador → modelo
PIPELINES = {
    nombre: Pipeline(steps=[
        ('preprocesador', preprocesador),
        ('modelo', modelo)
    ])
    for nombre, modelo in MODELOS.items()
}

print(f'  {len(PIPELINES)} pipelines definidos.')
print('  IsotonicRegression: excluida (requiere entrada 1D univariada).')

## 5. Validación Cruzada (K-Fold, k=5)

La validación cruzada repite la evaluación k veces con diferentes divisiones,
devolviendo métricas más robustas que una única evaluación train/test.

**Métricas calculadas:**
- **MAE** (Mean Absolute Error): promedio de |real − predicho|. Interpretable en unidades del precio ($).
- **RMSE** (Root Mean Squared Error): penaliza más los errores grandes.
- **R²**: proporción de variabilidad del precio explicada por el modelo (1 = perfecto, 0 = sin capacidad predictiva).

Se reporta `µ ± σ` para MAE y RMSE, y `µ` para R².

In [ ]:
print('[5/8] Ejecutando Validación Cruzada (K=5)...')
print('      (Esto puede tomar varios minutos)\n')

kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv = {}

for nombre, pipeline in PIPELINES.items():
    print(f'  → {nombre}...', end=' ', flush=True)
    cv_res = cross_validate(
        pipeline, X_train, y_train, cv=kf,
        scoring={
            'MAE':  'neg_mean_absolute_error',
            'RMSE': 'neg_root_mean_squared_error',
            'R2':   'r2'
        },
        return_train_score=False, n_jobs=-1
    )
    mae_s  = -cv_res['test_MAE']
    rmse_s = -cv_res['test_RMSE']
    r2_s   =  cv_res['test_R2']
    resultados_cv[nombre] = {
        'MAE_mean':  mae_s.mean(),  'MAE_std':  mae_s.std(),
        'RMSE_mean': rmse_s.mean(), 'RMSE_std': rmse_s.std(),
        'R2_mean':   r2_s.mean(),
    }
    print(f'MAE={mae_s.mean():.2f}±{mae_s.std():.2f} | RMSE={rmse_s.mean():.2f} | R²={r2_s.mean():.3f}')

In [ ]:
# Tabla comparativa
tabla_cv = pd.DataFrame(resultados_cv).T.sort_values('RMSE_mean')

print('\n' + '=' * 65)
print('  TABLA COMPARATIVA — VALIDACIÓN CRUZADA (K=5)')
print('=' * 65)
display(tabla_cv.style
    .format('{:.4f}')
    .background_gradient(subset=['RMSE_mean'], cmap='RdYlGn_r')
    .background_gradient(subset=['R2_mean'],   cmap='RdYlGn')
)

modelo_estable  = tabla_cv['RMSE_std'].idxmin()
modelo_variable = tabla_cv['RMSE_std'].idxmax()
mejor_modelo_cv = tabla_cv['RMSE_mean'].idxmin()

print(f'\n  ✔ Modelo más estable (menor σ RMSE): {modelo_estable}')
print(f'  ✗ Modelo con mayor varianza (σ RMSE): {modelo_variable}')
print(f'  ★ Mejor modelo (RMSE_mean):           {mejor_modelo_cv}')

## 6. Ajuste de Hiperparámetros

Se seleccionan dos modelos para ajuste:
- **Lineal:** Ridge (`GridSearchCV` — búsqueda exhaustiva, espacio pequeño)
- **No lineal:** LightGBM (`RandomizedSearchCV` — espacio grande, muestreo aleatorio)

**Corrección L4-2:** El modelo no lineal es **LightGBM**, no Random Forest.
El script original etiquetaba `gs_lgbm` como "Random Forest" en 3 lugares.

**Corrección L4-4:** Se añade `R2` al `scoring` de `RandomizedSearchCV` y se extrae
directamente de `cv_results_`, eliminando la llamada redundante a `cross_val_score`.

In [ ]:
print('[6/8] Ajuste de Hiperparámetros...')

# ── 6.1 Ridge — GridSearchCV ──────────────────────────────────────────────
print('\n  Ajustando Ridge (modelo lineal)...')

param_grid_ridge = {'modelo__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

gs_ridge = GridSearchCV(
    PIPELINES['Ridge'], param_grid_ridge,
    cv=kf, scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=0
)
gs_ridge.fit(X_train, y_train)

mejor_alpha_ridge = gs_ridge.best_params_['modelo__alpha']
mejor_rmse_ridge  = -gs_ridge.best_score_

print(f'    Mejor alpha:              {mejor_alpha_ridge}')
print(f'    RMSE CV (Ridge ajustado): {mejor_rmse_ridge:.4f}')
print(f'    RMSE CV (Ridge base):     {resultados_cv["Ridge"]["RMSE_mean"]:.4f}')

# ── 6.2 LightGBM — RandomizedSearchCV (FIX L4-2 + L4-4) ──────────────────
print('\n  Ajustando LightGBM (modelo no lineal de boosting por árboles)...')
print('  (FIX L4-2: corregido — antes estaba etiquetado como "Random Forest")')

param_grid_lgbm = {
    'modelo__n_estimators':      [100, 200, 300],
    'modelo__max_depth':         [-1, 6, 10],
    'modelo__learning_rate':     [0.05, 0.1, 0.2],
    'modelo__num_leaves':        [31, 63, 127],
    'modelo__min_child_samples': [20, 50],
}

# FIX L4-4: añadir R2 al scoring para evitar cross_val_score redundante después
gs_lgbm = RandomizedSearchCV(
    PIPELINES['LightGBM'], param_grid_lgbm,
    n_iter=20, cv=kf,
    scoring={'RMSE': 'neg_root_mean_squared_error', 'R2': 'r2'},
    refit='RMSE',
    n_jobs=-1, random_state=42, verbose=0
)
gs_lgbm.fit(X_train, y_train)

mejor_params_lgbm = gs_lgbm.best_params_
mejor_rmse_lgbm   = -gs_lgbm.best_score_

# FIX L4-4: R² directamente de cv_results_ (sin cross_val_score adicional)
mejor_r2_lgbm = gs_lgbm.cv_results_['mean_test_R2'][gs_lgbm.best_index_]

print(f'    Mejores hiperparámetros:         {mejor_params_lgbm}')
print(f'    RMSE CV (LightGBM ajustado):     {mejor_rmse_lgbm:.4f}')
print(f'    RMSE CV (LightGBM base):         {resultados_cv["LightGBM"]["RMSE_mean"]:.4f}')
print(f'    R²   CV (LightGBM ajustado):     {mejor_r2_lgbm:.4f}')
print(f'    (FIX L4-4: R² extraído de cv_results_, no de cross_val_score redundante)')

## 7. Evaluación Final — Hold-Out Test Set

Se evalúa el mejor modelo encontrado sobre el conjunto de **test** (30% de los datos).
Este conjunto nunca fue visto durante el entrenamiento ni la búsqueda de hiperparámetros.

**Corrección L4-2:** Los candidatos finales están correctamente etiquetados.
`gs_lgbm` es **LightGBM**, no Random Forest.

In [ ]:
print('[7/8] Evaluación Final en Test (Hold-Out)...')

# FIX L4-2: candidatos correctamente etiquetados
candidatos_finales = {
    'Ridge (ajustado)':    (gs_ridge.best_estimator_, mejor_rmse_ridge),
    'LightGBM (ajustado)': (gs_lgbm.best_estimator_,  mejor_rmse_lgbm),
}

# Añadir el mejor modelo de CV si no es Ridge ni LightGBM
if mejor_modelo_cv not in ['Ridge', 'LightGBM']:
    pipe_cv = PIPELINES[mejor_modelo_cv]
    pipe_cv.fit(X_train, y_train)
    rmse_temp = np.sqrt(mean_squared_error(y_test, pipe_cv.predict(X_test)))
    candidatos_finales[f'{mejor_modelo_cv} (CV)'] = (pipe_cv, rmse_temp)

nombre_ganador = min(candidatos_finales, key=lambda k: candidatos_finales[k][1])
modelo_final, _ = candidatos_finales[nombre_ganador]
print(f'  Modelo seleccionado: {nombre_ganador}')

y_pred_test = modelo_final.predict(X_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test   = r2_score(y_test, y_pred_test)

print('\n  --- RESULTADOS EN TEST (HOLD-OUT) ---')
print(f'    MAE : ${mae_test:.2f}  (error absoluto promedio por noche)')
print(f'    RMSE: ${rmse_test:.2f}  (penaliza errores grandes)')
print(f'    R²  : {r2_test:.4f}  ({r2_test*100:.1f}% de variabilidad explicada)')

cv_rmse = candidatos_finales[nombre_ganador][1]
dif = rmse_test - cv_rmse
print(f'\n    RMSE CV (entrenamiento): ${cv_rmse:.2f}')
print(f'    RMSE Test (producción):  ${rmse_test:.2f}')
if abs(dif) / cv_rmse < 0.10:
    print('    ✔ El modelo GENERALIZA BIEN (diferencia < 10%)')
else:
    print(f'    ⚠  Diferencia: ${dif:.2f} → posible sobreajuste')

## 9. Análisis de Residuos

Los residuos = (precio real − precio predicho). Un buen modelo de regresión debe tener:
1. **Homocedasticidad:** varianza constante en los residuos (sin forma de embudo).
2. **Distribución centrada en 0:** sin sesgo sistemático.
3. **Sin patrón no aleatorio:** los errores no deben seguir una curva o tendencia.

**Corrección L4-1:** Las figuras se guardan en `REPORTS_DIR` (ruta absoluta),
no en el CWD del proceso.

In [ ]:
print('[8/8] Generando figuras...')
residuos = y_test.values - y_pred_test

# ── Figura 1: Tabla comparativa de modelos ───────────────────────────────
fig1, ax1 = plt.subplots(figsize=(12, 5))
ax1.axis('off')
tabla_display = tabla_cv.copy().round(3)
tabla_display.columns = ['MAE µ', 'MAE σ', 'RMSE µ', 'RMSE σ', 'R² µ']
tbl = ax1.table(
    cellText=tabla_display.values,
    rowLabels=tabla_display.index,
    colLabels=tabla_display.columns,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.8)
for i, nm in enumerate(tabla_display.index):
    color = '#d1fae5' if nm == mejor_modelo_cv else 'white'
    for j in range(len(tabla_display.columns)):
        tbl[i + 1, j].set_facecolor(color)
ax1.set_title('Comparación de Modelos – Validación Cruzada (K=5)\n'
              '(Fila verde = mejor modelo por RMSE)', fontsize=12, pad=20)
plt.tight_layout()
# FIX L4-1: ruta absoluta a REPORTS_DIR
fig1.savefig(REPORTS_DIR / 'fig1_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 1 guardada en: {REPORTS_DIR / "fig1_comparacion_modelos.png"}')

In [ ]:
# ── Figura 2: Métricas por modelo (barras) ───────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle('Métricas de Validación Cruzada por Modelo', fontsize=13, fontweight='bold')

metricas_plot = [
    ('RMSE_mean', 'RMSE_std', 'RMSE Promedio (±σ)', 'salmon'),
    ('MAE_mean',  'MAE_std',  'MAE Promedio (±σ)',  'steelblue'),
    ('R2_mean',   None,       'R² Promedio',         'seagreen'),
]
for ax, (col_m, col_s, titulo, color) in zip(axes2, metricas_plot):
    valores  = tabla_cv[col_m]
    nombres  = [n.replace(' ', '\n') for n in tabla_cv.index]
    errores  = tabla_cv[col_s] if col_s else None
    ax.barh(nombres, valores, xerr=errores, color=color, alpha=0.8, capsize=4, edgecolor='white')
    ax.set_xlabel(titulo, fontsize=10)
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.invert_yaxis()

plt.tight_layout()
fig2.savefig(REPORTS_DIR / 'fig2_metricas_cv.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 2 guardada en: {REPORTS_DIR / "fig2_metricas_cv.png"}')

In [ ]:
# ── Figura 3: Análisis de residuos ───────────────────────────────────────
fig3, axes3 = plt.subplots(2, 2, figsize=(13, 10))
fig3.suptitle(f'Análisis de Residuos – {nombre_ganador}', fontsize=13, fontweight='bold')

# 3a: Residuos vs Predichos
ax = axes3[0, 0]
ax.scatter(y_pred_test, residuos, alpha=0.4, s=15, color=PALETTE, edgecolors='none')
ax.axhline(0, color='red', linewidth=1.5, linestyle='--', label='Residuo=0')
ax.set_xlabel('Precio Predicho ($)'); ax.set_ylabel('Residuo (Real − Predicho)')
ax.set_title('Residuos vs Predicciones\n(patrón aleatorio = homocedasticidad)')
ax.legend(fontsize=8)

# 3b: Histograma de residuos
ax = axes3[0, 1]
ax.hist(residuos, bins=50, color=PALETTE, alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linewidth=1.5, linestyle='--')
ax.axvline(residuos.mean(), color='orange', linewidth=1.5, linestyle='-',
           label=f'Media: ${residuos.mean():.2f}')
ax.set_xlabel('Residuo ($)'); ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de Residuos\n(ideal: campana centrada en 0)')
ax.legend(fontsize=8)

# 3c: Real vs Predicho
ax = axes3[1, 0]
lim_min = min(float(y_test.min()), float(y_pred_test.min()))
lim_max = max(float(y_test.max()), float(y_pred_test.max()))
ax.scatter(y_test, y_pred_test, alpha=0.4, s=15, color='seagreen', edgecolors='none')
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=1.5, label='Predicción perfecta')
ax.set_xlabel('Precio Real ($)'); ax.set_ylabel('Precio Predicho ($)')
ax.set_title(f'Real vs Predicho\nR²={r2_test:.3f} | RMSE=${rmse_test:.2f}')
ax.legend(fontsize=8)

# 3d: Residuos vs Índice (autocorrelación)
ax = axes3[1, 1]
ax.scatter(range(len(residuos)), residuos, alpha=0.3, s=10, color='purple', edgecolors='none')
ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
ax.set_xlabel('Índice de Observación'); ax.set_ylabel('Residuo ($)')
ax.set_title('Residuos vs Índice\n(detecta autocorrelación / tendencia temporal)')

plt.tight_layout()
fig3.savefig(REPORTS_DIR / 'fig3_analisis_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 3 guardada en: {REPORTS_DIR / "fig3_analisis_residuos.png"}')

## 8. Interpretabilidad del Modelo

**Corrección L4-3:** El script original obtenía los nombres de features del preprocesador
de **Ridge** para graficar las importancias de **LightGBM**. Esto causaba un mismatch
potencial si el número de categorías OHE difería entre ambos pipelines.

La corrección extrae los nombres directamente del preprocesador del pipeline de LightGBM.

In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(14, 6))
fig4.suptitle('Interpretabilidad del Modelo Final', fontsize=13, fontweight='bold')

try:
    # ── Asegurar que los pipelines están ajustados ────────────────────────
    ridge_pipe = gs_ridge.best_estimator_
    lgbm_pipe  = gs_lgbm.best_estimator_
    if not hasattr(ridge_pipe.named_steps['preprocesador'], 'transformers_'):
        ridge_pipe.fit(X_train, y_train)
    if not hasattr(lgbm_pipe.named_steps['preprocesador'], 'transformers_'):
        lgbm_pipe.fit(X_train, y_train)

    # ── Coeficientes Ridge ────────────────────────────────────────────────
    ohe_ridge = (ridge_pipe.named_steps['preprocesador']
                 .named_transformers_['cat'].named_steps['codificar'])
    nombres_cat_ridge = ohe_ridge.get_feature_names_out(VARIABLES_CATEGORICAS).tolist()
    nombres_ridge = VARIABLES_NUMERICAS + nombres_cat_ridge

    coefs = ridge_pipe.named_steps['modelo'].coef_
    n_c   = min(len(coefs), len(nombres_ridge))
    coefs, nombres_ridge = coefs[:n_c], nombres_ridge[:n_c]
    orden_r = np.argsort(np.abs(coefs))[-15:]
    colores_r = ['#ef4444' if c < 0 else '#22c55e' for c in coefs[orden_r]]

    axes4[0].barh([nombres_ridge[i] for i in orden_r], coefs[orden_r],
                  color=colores_r, edgecolor='white')
    axes4[0].axvline(0, color='black', linewidth=1)
    axes4[0].set_title('Coeficientes Ridge (Top 15)\nVerde=+precio, Rojo=−precio', fontsize=9)
    axes4[0].set_xlabel('Coeficiente (impacto en precio USD)')

    # ── FIX L4-3: Feature importances de LightGBM ────────────────────────
    # Nombres extraídos del preprocesador de LightGBM, NO del de Ridge
    ohe_lgbm = (lgbm_pipe.named_steps['preprocesador']
                .named_transformers_['cat'].named_steps['codificar'])
    nombres_cat_lgbm  = ohe_lgbm.get_feature_names_out(VARIABLES_CATEGORICAS).tolist()
    nombres_lgbm = VARIABLES_NUMERICAS + nombres_cat_lgbm

    importancias = lgbm_pipe.named_steps['modelo'].feature_importances_
    n_i = min(len(importancias), len(nombres_lgbm))
    importancias, nombres_lgbm = importancias[:n_i], nombres_lgbm[:n_i]
    orden_l = np.argsort(importancias)[-15:]

    # FIX L4-2: título corregido — LightGBM, no Random Forest
    axes4[1].barh([nombres_lgbm[i] for i in orden_l], importancias[orden_l],
                  color='steelblue', edgecolor='white', alpha=0.85)
    axes4[1].set_title('Importancia de Variables – LightGBM (Top 15)\n'
                       'Mayor valor = más influyente en el precio', fontsize=9)
    axes4[1].set_xlabel('Importancia relativa (gain)')

except Exception as e:
    print(f'  ⚠ No se pudo generar gráfico de importancia: {e}')
    for ax in axes4:
        ax.text(0.5, 0.5, 'No disponible', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
fig4.savefig(REPORTS_DIR / 'fig4_importancia_variables.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 4 guardada en: {REPORTS_DIR / "fig4_importancia_variables.png"}')
print(f'  (FIX L4-2: título corregido a LightGBM)')
print(f'  (FIX L4-3: nombres de features de LightGBM, no de Ridge)')

## 10. Selección del Modelo Final

El modelo final se elige considerando tres criterios combinados:

| Criterio | Peso |
|---|---|
| Métricas en validación cruzada (RMSE, R²) | Alto |
| Desempeño en test (generalización) | Alto |
| Interpretabilidad | Medio |

> **Nota:** Un RMSE en CV muy distinto al RMSE en test indica sobreajuste.
> Se prefiere el modelo con menor brecha CV−Test sobre el de menor RMSE absoluto.

In [ ]:
print('\n  --- SELECCIÓN DEL MODELO FINAL ---')
print(f'  Mejor modelo por CV:    {mejor_modelo_cv}')
print(f'  Modelo ganador (test):  {nombre_ganador}')
print(f'  MAE  test: ${mae_test:.2f}')
print(f'  RMSE test: ${rmse_test:.2f}')
print(f'  R²   test: {r2_test:.4f}')

# Tabla resumen CV vs Test
resumen = pd.DataFrame({
    'RMSE CV':   [candidatos_finales[nombre_ganador][1]],
    'RMSE Test': [rmse_test],
    'MAE Test':  [mae_test],
    'R² Test':   [r2_test],
}, index=[nombre_ganador])
display(resumen.round(4))

## 11. Guardado del Modelo

**Corrección L4-1:** El script original usaba `'../models'` (ruta relativa al CWD),
lo que guardaba el modelo fuera del proyecto si el CWD era el directorio raíz.
Aquí se usa la ruta absoluta `MODELS_DIR` derivada al inicio del notebook.

In [ ]:
print('[+] Guardando modelo final...')

model_path    = MODELS_DIR / 'model_regression.joblib'
features_path = MODELS_DIR / 'features_regression.joblib'

# FIX L4-1: rutas absolutas — MODELS_DIR = PROJECT_DIR / 'models'
joblib.dump(modelo_final, model_path)
joblib.dump(VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS, features_path)

print(f'  ✔ Modelo guardado en:    {model_path}')
print(f'  ✔ Features guardadas en: {features_path}')

# Verificación de carga
loaded = joblib.load(model_path)
sample_pred = loaded.predict(X_test.iloc[:3])
print(f'\n  Verificación — predicciones primeras 3 muestras: {sample_pred.round(2)}')
print('  Carga exitosa.')

## Resumen Ejecutivo

| Ítem | Detalle |
|---|---|
| Dataset | Airbnb listings |
| Variable objetivo | `price` (USD/noche) |
| División | 70% train / 30% test |
| Validación | K-Fold (k=5) |
| Modelos entrenados | 7 (excl. IsotonicRegression) |
| Tuning | GridSearchCV (Ridge) + RandomizedSearchCV (LightGBM) |
| Figuras | `reports/lab4/fig1-4_*.png` |
| Modelos | `models/model_regression.joblib` |

**Correcciones aplicadas:**
- L4-1: rutas absolutas con `pathlib` (figuras y modelos en directorios correctos)
- L4-2: LightGBM correctamente etiquetado en candidatos, figura y comentarios
- L4-3: feature importances extraídas del preprocesador de LightGBM
- L4-4: R² de `cv_results_`, eliminada llamada a `cross_val_score` redundante
- L4-5: variables adicionales (`neighbourhood_group`, etc.) incluidas si existen